# Aurora Siger — Sistema de Análise de Telemetria

Sistema de verificação pré-lançamento com análise por IA (Google Gemini).

> **Pré-requisito:** Adicione sua chave em **Secrets** (ícone de chave no painel esquerdo) com o nome `GEMINI_API_KEY` antes de executar.

## 1. Instalação

Instala o SDK oficial do Gemini (`google-genai`).

In [ ]:
!pip install -q google-genai

## 2. Configuração do cliente Gemini

Lê a chave de API diretamente dos **Secrets** do Colab e inicializa o cliente.

In [ ]:
from google.colab import userdata
from google import genai

cliente = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print('Cliente Gemini configurado.')

## 3. Dados de telemetria

Preencha os valores quando solicitado. Faixas válidas:

| Parâmetro | Faixa válida |
|---|---|
| Temperatura interna | -10°C a 40°C |
| Temperatura externa | -5°C a 45°C |
| Energia | ≥ 70% |
| Combustível RP-1 | ≥ 80% |
| Oxidante LOX | ≥ 80% |
| Pressão do tanque | 2,5 a 4,5 bar |
| Integridade estrutural | 1 = OK, 0 = ERRO |

In [ ]:
# INPUTS
temp_interna      = float(input('Temperatura interna (-10 a 40°C): '))
temp_externa      = float(input('Temperatura externa (-5 a 45°C): '))
nivel_energia     = float(input('Nível de energia (% mínimo 70): '))
nivel_combustivel = float(input('Nível de combustível RP-1 (% mínimo 80): '))
nivel_oxidante    = float(input('Nível de oxidante LOX (% mínimo 80): '))
pressao_tanque    = float(input('Pressão do tanque (2.5 a 4.5 bar): '))
integridade       = int(input('Integridade estrutural (1 = OK, 0 = ERRO): '))

## 4. Verificação de sistemas

Valida cada parâmetro contra os limites operacionais e exibe o status de lançamento.

In [ ]:
import time

# CÁLCULOS
gradiente_termico = abs(temp_interna - temp_externa)

print('\n[ Analisando sistema... ]\n')
time.sleep(0.5)

# VERIFICAÇÕES
resultados = []

if not (-10 <= temp_interna <= 40):
    resultados.append(('Temperatura interna', False, f'Fora do limite ({temp_interna}°C). Esperado: -10°C a 40°C.'))
else:
    resultados.append(('Temperatura interna', True, None))

if not (-5 <= temp_externa <= 45):
    resultados.append(('Temperatura externa', False, f'Fora do limite ({temp_externa}°C). Esperado: -5°C a 45°C.'))
else:
    resultados.append(('Temperatura externa', True, None))

if not (gradiente_termico <= 40):
    resultados.append(('Gradiente térmico', False, f'Gradiente excessivo ({gradiente_termico:.1f}°C). Máx: 40°C.'))
else:
    resultados.append(('Gradiente térmico', True, None))

if not (nivel_energia >= 70):
    resultados.append(('Nível de energia', False, f'Energia insuficiente ({nivel_energia:.1f}%). Mín: 70%.'))
else:
    resultados.append(('Nível de energia', True, None))

if not (nivel_combustivel >= 80):
    resultados.append(('Nível de combustível (RP-1)', False, f'Combustível insuficiente ({nivel_combustivel:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de combustível (RP-1)', True, None))

if not (nivel_oxidante >= 80):
    resultados.append(('Nível de oxidante (LOX)', False, f'Oxidante insuficiente ({nivel_oxidante:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de oxidante (LOX)', True, None))

if not (2.5 <= pressao_tanque <= 4.5):
    resultados.append(('Pressão do tanque', False, f'Pressão fora do intervalo ({pressao_tanque:.2f} bar). Esperado: 2.5–4.5 bar.'))
else:
    resultados.append(('Pressão do tanque', True, None))

if not (integridade == 1):
    resultados.append(('Integridade estrutural', False, 'Falha estrutural detectada.'))
else:
    resultados.append(('Integridade estrutural', True, None))

status_modulos = all(passou for _, passou, _ in resultados)
resultados.append(('Status dos módulos', status_modulos, 'Módulos com falha detectada.'))

for nome, passou, erro in resultados:
    if not passou:
        print(f'  >> {nome}: FALHA')
        print(f'     {erro}')
        print('\n❌ DECOLAGEM ABORTADA')
        break
    print(f'  >> {nome}: OK')
    time.sleep(0.7)
else:
    print('\n✅ PRONTO PARA DECOLAR')

## 5. Análise IA — Gemini

Envia a telemetria ao Gemini para análise de risco e classificação de anomalias.

In [ ]:
# ANÁLISE IA
import textwrap

status_texto = 'PRONTO PARA DECOLAR' if status_modulos else 'DECOLAGEM ABORTADA'
integridade_texto = 'OK' if integridade == 1 else 'ERRO'

prompt = f"""
Você é um sistema de análise de telemetria aeroespacial. Analise os dados da Missão Aurora Siger:

TELEMETRIA:
- Temperatura interna: {temp_interna}°C
- Temperatura externa: {temp_externa}°C
- Gradiente térmico: {gradiente_termico:.1f}°C
- Nível de energia: {nivel_energia:.1f}%
- Nível de combustível (RP-1): {nivel_combustivel:.1f}%
- Nível de oxidante (LOX): {nivel_oxidante:.1f}%
- Pressão do tanque: {pressao_tanque:.2f} bar
- Integridade estrutural: {integridade_texto}
- Status geral: {status_texto}

Responda em português com:
1. Classificação de cada dado (normal / atenção / crítico)
2. Possíveis anomalias identificadas
3. Sugestões de risco
4. Formato de resposta: Texto simples, direto e claro. Sem formatação adicional.
"""

print('[ Análise IA - Gemini ]')
print('\n[ Aguardando resultado... ]\n')
resposta = cliente.models.generate_content(model='gemini-2.5-flash', contents=prompt)

for linha in resposta.text.splitlines():
    print(textwrap.fill(linha, width=100) if linha.strip() else '')
